In [3]:
import pandas as pd
import numpy as np
from datetime import datetime
from dotenv import load_dotenv
from highwayentrance_test import get_coordinates, get_route,  roadnames_in_route, is_highway, get_highway_inout_points, haversine, extract_variable_interval_points, visualize_route_with_5km

def stat_filtering(df):
    '''
    1개 이상 사용 가능(stat = 2 or 3)한 충전소만 필터링하는 함수
    input: 전처리를 마친 dataframe
    output: working_ratio(전체 충전기 중 stat = 2 or 3인 충전기의 비율)가 0보다 큰 충전소들의 dataframe
    '''
    return df[df['working_ratio'] > 0]

def recent_filtering(df, hours=48, now=None):
    '''
    현재 시점 기준 48시간 이내에 사용 기록이 있는지 여부 기반 필터링 함수
    input: 
        df: dataframe, 
        hours: 기준 시간(default 48시간), 
        now: 현재 시점을 임의로 부여 / 부여하지 않을 경우 현재 시간 불러와서 처리
    output: 현재 시점 기준 hours 이내에 사용 기록이 있는 충전소 dataframe
    '''
    if now is None:
        now = pd.Timestamp.now()
    
    # 최근 충전 '시작' 시간과 최근 충전 '종료' 시간 중 더 최근 시간을 기준으로 계산
    max_time = df[['lastTsdt', 'lastTedt']].max(axis=1)

    # 기준 시각에서 `hours` 이전보다 더 늦은 것만 남기기
    return df[max_time >= (now - pd.Timedelta(hours=hours))].copy()


def filtering_first(df, hours=48, now=None):
    '''
    stat, 시간 기준의 1차 필터링 함수
    input: dataframe, hours, now
    output: 1차 필터링을 거친 데이터프레임
    '''
    temp_df = stat_filtering(df)
    result_df = recent_filtering(temp_df, hours, now)
    return result_df

def filter_by_distance_vectorized(df, center, max_distance_km=5):
    '''
    거리 기반 m km 이내의 충전소만 필터링하는 함수
    input: 
        df: dataframe
        centor: (위도, 경도)
        max_distance_km: m km, default = 5 km
    '''
    R = 6371  # 지구 반지름 (단위: km)
    lon1, lat1 = np.radians(center)

    lat2 = np.radians(df['lat'].values)
    lon2 = np.radians(df['lng'].values)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    distances = R * c

    df = df.copy()
    df['distance_km'] = distances  

    return df.loc[distances <= max_distance_km]

def filter_by_user_input(df, 
                         output_values=None, 
                         chger_types=None, 
                         kinds=None, 
                         busi_ids=None):
    '''
    2차 필터링: 사용자 입력 변수 기반 필터링
    input:
        output: 충전 용량(3, 7, 50, 100, 200)
        chgertype: 충전기 커넥터 유형(01:DC차데모,02: AC완속,03: DC차데모+AC3상,04: DC콤보,05: DC차데모+DC콤보, 06: DC차데모+AC3상+DC콤보, 07: AC3상, 08: DC콤보(완속), 09: NACS, 10: DC콤보+NACS)
        kind: 관련 시설 종류(공공시설, 주차시설 등)
        busid: 충전 사업자(GS 칼텍스, 현대자동차 등)
    output:
        2차 필터링이 적용된 데이터프레임 
    '''
    filtered = df.copy()

    if output_values is not None:
        filtered = filtered[filtered['output'].isin(output_values)]
    
    if chger_types is not None:
        filtered = filtered[filtered['chgerType'].astype(str).isin(chger_types)]

    if kinds is not None:
        filtered = filtered[filtered['kind'].isin(kinds)]
    
    if busi_ids is not None:
        filtered = filtered[filtered['busiId'].isin(busi_ids)]

    return filtered

def score_stations(df, k=3):
    '''
    스코어링 함수, top k개 반환
    input: N km 지점마다의 point로부터 m km 이내의 충전소 데이터프레임
    output: 스코어 기반 top k개 dataframe
    '''
    df = df.copy()

    # 미리 전처리된 변수들 기준으로 스코어링에 필요한 이진 변수 생성
    df['is_free_parking'] = df['parkingFree'].apply(lambda x: 1 if x == 'Y' else 0)
    df['has_convenience'] = df['trafficYn'].apply(lambda x: 1 if x == 'Y' else 0)

    # 급속 여부: output 문자열을 쉼표로 구분된 값 중 하나라도 50 이상이면 1
    def is_fast_output(output_str):
        try:
            outputs = [float(o) for o in output_str.split(',') if o.strip().isdigit()]
            return int(any(o >= 50 for o in outputs))
        except:
            return 0

    df['is_fast'] = df['output'].apply(is_fast_output)

    # avg_cost 컬럼이 없으므로 가정
    df['avg_cost'] = 300
    df['inv_cost'] = 1 / (df['avg_cost'] + 1e-6)

    # 가중치 설정
    weights = {
        # 'is_open': 1.0,  # 제외됨
        'is_free_parking': 0.5,
        'working_ratio': 1.0,
        'congestion_ratio': -0.8,
        'inv_cost': 0.8,
        'has_convenience': 0.4,
        'is_fast': 0.4,
        'distance_km': -1.5
    }

    for feature, weight in weights.items():
        if weight >= 0:
            df[feature + '_score'] = df[feature] * weight
        else:
            # 혼잡도나 거리처럼 작을수록 좋은 경우는 (1 - x) * -w
            df[feature + '_score'] = (1 - df[feature]) * (-weight) if 'ratio' in feature else df[feature] * weight

    # 총합 스코어 계산
    score_cols = [col for col in df.columns if col.endswith('_score')]
    df['total_score'] = df[score_cols].sum(axis=1)

    return df.sort_values('total_score', ascending=False).head(k)


In [10]:
def score_highway(df, k=3):
    '''
    스코어링 함수, top k개 반환
    input: N km 지점마다의 point로부터 m km 이내의 충전소 데이터프레임
    output: 스코어 기반 top k개 dataframe
    '''
    df = df.copy()

    # 미리 전처리된 변수들 기준으로 스코어링에 필요한 이진 변수 생성
    df['is_free_parking'] = df['parkingFree'].apply(lambda x: 1 if x == 'Y' else 0)
    df['has_convenience'] = df['trafficYn'].apply(lambda x: 1 if x == 'Y' else 0)

    # 급속 여부: output 문자열을 쉼표로 구분된 값 중 하나라도 50 이상이면 1
    def is_fast_output(output_str):
        try:
            outputs = [float(o) for o in output_str.split(',') if o.strip().isdigit()]
            return int(any(o >= 50 for o in outputs))
        except:
            return 0

    df['is_fast'] = df['output'].apply(is_fast_output)

    # avg_cost 컬럼이 없으므로 가정
    df['avg_cost'] = 300
    df['inv_cost'] = 1 / (df['avg_cost'] + 1e-6)

    # 가중치 설정
    weights = {
        # 'is_open': 1.0,  # 제외됨
        'is_free_parking': 0.5,
        'working_ratio': 1.0,
        'congestion_ratio': -0.8,
        'inv_cost': 0.8,
        'has_convenience': 0.4,
        'is_fast': 0.4,
        'distance_km': -1.5
    }

    for feature, weight in weights.items():
        if weight >= 0:
            df[feature + '_score'] = df[feature] * weight
        else:
            # 혼잡도나 거리처럼 작을수록 좋은 경우는 (1 - x) * -w
            df[feature + '_score'] = (1 - df[feature]) * (-weight) if 'ratio' in feature else df[feature] * weight

    # 총합 스코어 계산
    score_cols = [col for col in df.columns if col.endswith('_score')]
    df['total_score'] = df[score_cols].sum(axis=1)

    return df.sort_values('total_score', ascending=False).head(k)


In [4]:
def score_start(df, k=3):
    '''
    출발(도착) 지점 근처의 스코어링 함수, top k개 반환
    input: N km 지점마다의 point로부터 m km 이내의 충전소 데이터프레임
    output: 스코어 기반 top k개 dataframe
    '''
    df = df.copy()

    # 미리 전처리된 변수들 기준으로 스코어링에 필요한 이진 변수 생성
    df['is_free_parking'] = df['parkingFree'].apply(lambda x: 1 if x == 'Y' else 0)
    df['has_convenience'] = df['trafficYn'].apply(lambda x: 1 if x == 'Y' else 0)

    # 급속 여부: output 문자열을 쉼표로 구분된 값 중 하나라도 50 이상이면 1
    def is_fast_output(output_str):
        try:
            outputs = [float(o) for o in output_str.split(',') if o.strip().isdigit()]
            return int(any(o >= 50 for o in outputs))
        except:
            return 0

    df['is_fast'] = df['output'].apply(is_fast_output)

    # avg_cost 컬럼이 없으므로 가정
    df['avg_cost'] = 300
    df['inv_cost'] = 1 / (df['avg_cost'] + 1e-6)

    # 가중치 설정
    weights = {
        # 'is_open': 1.0,  # 제외됨
        'is_free_parking': 0.5,
        'working_ratio': 1.0,
        'congestion_ratio': -0.8,
        'inv_cost': 0.8,
        'has_convenience': 0.4,
        'is_fast': 0.4,
        'distance_km': -1.5
    }

    for feature, weight in weights.items():
        if weight >= 0:
            df[feature + '_score'] = df[feature] * weight
        else:
            # 혼잡도나 거리처럼 작을수록 좋은 경우는 (1 - x) * -w
            df[feature + '_score'] = (1 - df[feature]) * (-weight) if 'ratio' in feature else df[feature] * weight

    # 총합 스코어 계산
    score_cols = [col for col in df.columns if col.endswith('_score')]
    df['total_score'] = df[score_cols].sum(axis=1)

    return df.sort_values('total_score', ascending=False).head(k)

In [40]:
def score_inout(df, k=3):
    '''
    고속도로 진출입로 근처의 스코어링 함수, top k개 반환
    input: N km 지점마다의 point로부터 m km 이내의 충전소 데이터프레임
    output: 스코어 기반 top k개 dataframe
    '''
    df = df.copy()

    # 미리 전처리된 변수들 기준으로 스코어링에 필요한 이진 변수 생성
    df['is_free_parking'] = df['parkingFree'].apply(lambda x: 1 if x == 'Y' else 0)
    df['has_convenience'] = df['trafficYn'].apply(lambda x: 1 if x == 'Y' else 0)

    # 급속 여부: output 문자열을 쉼표로 구분된 값 중 하나라도 50 이상이면 1
    def is_fast_output(output_str):
        try:
            outputs = [float(o) for o in output_str.split(',') if o.strip().isdigit()]
            return int(any(o >= 50 for o in outputs))
        except:
            return 0

    df['is_fast'] = df['output'].apply(is_fast_output)

    # avg_cost 컬럼이 없으므로 가정
    df['avg_cost'] = 300
    df['inv_cost'] = 1 / (df['avg_cost'] + 1e-6)

    # 가중치 설정
    weights = {
        # 'is_open': 1.0,  # 제외됨
        'is_free_parking': 0.5,
        'working_ratio': 1.0,
        'congestion_ratio': -0.8,
        'inv_cost': 0.8,
        'has_convenience': 0.4,
        'is_fast': 0.4,
        'distance_km': -10
    }

    for feature, weight in weights.items():
        if weight >= 0:
            df[feature + '_score'] = df[feature] * weight
        else:
            # 혼잡도나 거리처럼 작을수록 좋은 경우는 (1 - x) * -w
            df[feature + '_score'] = (1 - df[feature]) * (-weight) if 'ratio' in feature else df[feature] * weight

    # 총합 스코어 계산
    score_cols = [col for col in df.columns if col.endswith('_score')]
    df['total_score'] = df[score_cols].sum(axis=1)

    return df.sort_values('total_score', ascending=False).head(k)

In [83]:
import pandas as pd
from datetime import datetime
from data_prep import data_load, preprocess_station_data
from scoring import (
    filtering_first,
    filter_by_distance_vectorized,
    filter_by_user_input,
    score_stations
)

def run_recommendation(
    data_dir: str,
    meta: dict,
    # center: tuple,
    max_distance_km: float,
    output_values: list,
    chger_types: list,
    kinds: list,
    busi_ids: list,
    k: int = 5,
    hours: int = 48
):
    
    center = (meta["lon"], meta["lat"])
    
    # # Step 1: 데이터 불러오기
    # raw_df = data_load(data_dir)

    # # Step 2: 전처리 (충전기 단위 → 충전소 단위)
    # station_df = preprocess_station_data(raw_df)
    # station_df.to_csv(f"../Database/preprocessed/processed_station_data.csv", index=False)

    station_df = pd.read_csv(f"../Database/preprocessed/processed_station_data.csv")

    # Step 3: 1차 필터링 (작동 상태 + 최근 사용 여부)
    # filtered_df = filtering_first(station_df, hours=hours, now=pd.Timestamp(datetime(2025, 5, 16, 12, 0, 0)))
    filtered_df = station_df

    # Step 4: 거리 필터링
    nearby_df = filter_by_distance_vectorized(filtered_df, center, max_distance_km=max_distance_km)

    #Step 5: 사용자 입력 필터링
    # user_filtered_df = filter_by_user_input(
    #     nearby_df,
    #     output_values=output_values,
    #     chger_types=chger_types,
    #     kinds=kinds,
    #     busi_ids=busi_ids
    # )

    user_filtered_df = nearby_df

    # Step 6: 스코어링
    # top_k_df = score_stations(user_filtered_df, k=k)
    
    if meta.get("road_type") == 'highway':
        top_k_df = score_highway(user_filtered_df, k=k)
    elif meta.get("inout") == "yes":
        top_k_df = score_inout(user_filtered_df, k=k)
    else:
        top_k_df = score_stations(user_filtered_df, k=k)

    # Step 7: 출력
    pd.set_option('display.max_columns', None)
    print("\n🚗 추천 충전소 Top {} 🚗\n".format(k))
    print(top_k_df[['statNm', 'addr', 'total_score', 'distance_km'] + 
                   [col for col in top_k_df.columns if col.endswith('_score')]])
    
    filename = f"../Database/result2/recommendation_{center[0]:.4f}_{center[1]:.4f}.csv"
    top_k_df.to_csv(filename, index=False)

    return top_k_df

def run_recommendation_with_list(
    meta_list: list,
    data_dir: str,
    max_distance_km: float,
    output_values: list,
    chger_types: list,
    kinds: list,
    busi_ids: list,
    k: int = 5,
    hours: int = 48
):
    results = []
    for meta in meta_list:
        top_k_df = run_recommendation(
            data_dir=data_dir,
            meta=meta,
            max_distance_km=max_distance_km,
            output_values=output_values,
            chger_types=chger_types,
            kinds=kinds,
            busi_ids=busi_ids,
            k=k,
            hours=hours
        )
        results.append(top_k_df)
    return results

def get_recommendations_from_inputs(
    meta_list: list,
    data_dir: str,
    max_distance_km: float,
    output_values: list,
    chger_types: list,
    kinds: list,
    busi_ids: list,
    k: int = 5,
    hours: int = 48
):
    return run_recommendation_with_list(
        meta_list=meta_list,
        data_dir=data_dir,
        max_distance_km=max_distance_km,
        output_values=output_values,
        chger_types=chger_types,
        kinds=kinds,
        busi_ids=busi_ids,
        k=k,
        hours=hours
    )

# 🔽 경로별 추천 파일로부터 요약 파일 생성 함수
def save_summary_from_recommendation_files(center_list, result_dir="../Database/result2", summary_filename="summary_route_recommendations.csv"):
    summary_rows = []
    for center in center_list:
        file_path = f"{result_dir}/recommendation_{center['lon']:.4f}_{center['lat']:.4f}.csv"
        try:
            df = pd.read_csv(file_path)
            if not df.empty:
                top = df.iloc[0]
                summary_rows.append({
                    "statNm": top["statNm"],
                    "위도": top["lat"],
                    "경도": top["lng"],
                    "경로상 위도": center['lat'],
                    "경로상 경도": center['lon']
                })
        except Exception as e:
            print(f"⚠️ 파일 읽기 실패: {file_path} ({e})")

    summary_df = pd.DataFrame(summary_rows)
    summary_path = f"{result_dir}/{summary_filename}"
    summary_df.to_csv(summary_path, index=False)
    print(f"\n✅ 요약 파일 저장 완료: {summary_filename}")

def main(
        origin_address = "서울특별시 서대문구 연세로 50",
        destination_address = "부산시 동래구 사직로 55-32",
        road_km = 10,
        highway_km = 50
    ):
    

    data_dir = '../Database/opendata'
    max_distance_km = 100
    output_values = ['50', '100', '200']
    chger_types = ['01', '02', '03', '04', '05', '06', '07', '08']
    kinds = ['A0', 'B0', 'C0']
    busi_ids = None #['ME', 'HI']
    k = 5

    # ✅ 1. 주소 → 좌표
    origin = get_coordinates(origin_address)
    destination = get_coordinates(destination_address)

    # ✅ 2. 경로 및 점 추출
    route = get_route(origin, destination)
    meta_list = extract_variable_interval_points(route, road_km, highway_km)

    # ✅ 3. 추천 수행
    results = get_recommendations_from_inputs(
        meta_list=meta_list,
        data_dir=data_dir,
        max_distance_km=max_distance_km,
        output_values=output_values,
        chger_types=chger_types,
        kinds=kinds,
        busi_ids=busi_ids,
        k=k
    )

    print("\n🚗 추천 충전소 최종결과 🚗\n")
    print(results)
    print("\n📁 추천 결과가 각 위치별로 CSV 파일로 저장되었습니다.")

    # 🔄 경로 요약 결과 통합 저장
    save_summary_from_recommendation_files(meta_list)



# if __name__ == '__main__':
#     main()




In [23]:
import folium
from folium.plugins import MarkerCluster

def visualize_recommendations_on_map(origin, destination, meta_list, recommendation_df, route=None, show=True):
    m = folium.Map(location=[origin[1], origin[0]], zoom_start=8)

    # 출발지
    folium.Marker([origin[1], origin[0]], popup="출발지", icon=folium.Icon(color='blue')).add_to(m)

    # 도착지
    folium.Marker([destination[1], destination[0]], popup="도착지", icon=folium.Icon(color='red')).add_to(m)

    # 경유 포인트 (meta_list)
    for pt in meta_list:
        folium.CircleMarker(
            location=[pt['lat'], pt['lon']],
            radius=3,
            color='gray',
            fill=True,
            popup=pt.get('name', 'point')
        ).add_to(m)

    # 추천 충전소
    for _, row in recommendation_df.iterrows():
        folium.Marker(
            [row['위도'], row['경도']],
            popup=row['statNm'],
            icon=folium.Icon(color='green', icon='flash')
        ).add_to(m)

    # 경로 표시
    if route:
        folium.PolyLine(
            [(lat, lon) for lon, lat in route],
            color='blue',
            weight=4,
            opacity=0.7
        ).add_to(m)

    # 지도 출력
    if show:
        return m
    else:
        m.save("recommendation_map.html")
        print("✅ 지도 저장 완료")


In [52]:
def visualize_recommendations_on_map(origin,road_km, highway_km, destination, route_data, waypoints=None, recommended_stations=None):
    """Folium 지도 위에 경로 + 5km마다 마커 + 추천 충전소 시각화"""
    m = folium.Map(location=[origin[0], origin[1]], zoom_start=14)

    # 출발지, 도착지
    folium.Marker([origin[0], origin[1]], tooltip="출발지", icon=folium.Icon(color='green')).add_to(m)
    folium.Marker([destination[0], destination[1]], tooltip="도착지", icon=folium.Icon(color='red')).add_to(m)

    # 경유지
    if waypoints:
        for idx, wp in enumerate(waypoints):
            folium.Marker([wp[0], wp[1]], tooltip=f"경유지 {idx+1}", icon=folium.Icon(color='blue')).add_to(m)

    # 경로 PolyLine
    sections = route_data['routes'][0]['sections']
    for section in sections:
        for road in section['roads']:
            coords = road['vertexes']
            points = [(coords[i+1], coords[i]) for i in range(0, len(coords), 2)]
            folium.PolyLine(points, color='black', weight=3).add_to(m)

    # N km 간격 마커
    interval_points = extract_variable_interval_points(route_data, road_km, highway_km)
    for i, pt in enumerate(interval_points):
        folium.CircleMarker(
            location=[pt['lat'], pt['lon']],
            radius=4,
            color='red' if pt.get('inout') == 'in' else 'blue' if pt.get('inout') == 'out' else 'purple',
            fill=True,
            fill_color='purple',
            fill_opacity=1,
            tooltip=f"{'[진입지점!] ' if pt.get('inout')=='in' else '[출구지점!] ' if pt.get('inout')=='out' else ''}{pt.get('name')}"
        ).add_to(m)

    # 🔋 충전소 마커
    if recommended_stations is not None:
        for idx, row in recommended_stations.iterrows():
            folium.Marker(
                location=[row['위도'], row['경도']],
                tooltip=f"[충전소] {row['statNm']}",
                icon=folium.Icon(color='orange', icon='flash')
            ).add_to(m)

    return m


In [94]:
start = "서울시 송파구 올림픽로 424"
# end = "경남 통영시 충렬로 33" 
end = "경북 경주시 포석로 1080"
main(origin_address=start, 
     destination_address=end,
     road_km=5,
     highway_km=50)


🚗 추천 충전소 Top 5 🚗

                  statNm                     addr  total_score  distance_km  \
14623               금복빌딩          서울 송파구 방이동 45-2     2.004107     0.465706   
22033    서울시 송파구 현대토픽스빌딩  서울특별시 송파구 위례성대로 6 (방이동)     1.583461     0.479470   
21819  서울시 송파구 잠실벨솔레오피스텔   서울특별시 송파구 올림픽로34길 5-13     1.375772     0.617930   
22012   서울시 송파구 올림픽공원파크텔       서울특별시 송파구 올림픽로 424     1.364475     0.292128   
16120              올림픽공원      서울특별시 송파구 올림픽로 424      1.252101     0.367044   

       is_free_parking_score  working_ratio_score  congestion_ratio_score  \
14623                    0.5                  1.0                     0.8   
22033                    0.5                  1.0                     0.8   
21819                    0.5                  1.0                     0.8   
22012                    0.0                  1.0                     0.8   
16120                    0.0                  1.0                     0.8   

       inv_cost_score  has_convenience_scor

In [95]:
origin = get_coordinates(start)
destination = get_coordinates(end)
road_km, highway_km = 5, 50
route = get_route(origin, destination)
meta_list = extract_variable_interval_points(route, road_km, highway_km)
results = pd.read_csv("../Database/result2/summary_route_recommendations.csv")

visualize_recommendations_on_map(origin,road_km, highway_km, destination, route, recommended_stations=results)

In [35]:
map_obj = visualize_recommendations_on_map(
    origin=origin,
    destination=destination,
    meta_list=meta_list,
    recommendation_df=results,
    route=route
)
map_obj


⚠️ 경로 시각화 실패: 0


In [31]:
print(route)

{'trans_id': '01975d332add7b1db533e01e23d2a080', 'routes': [{'result_code': 0, 'result_msg': '길찾기 성공', 'summary': {'origin': {'name': '', 'x': 127.02928413452449, 'y': 37.495549035548684}, 'destination': {'name': '', 'x': 128.41815052326402, 'y': 34.844606372446954}, 'waypoints': [], 'priority': 'RECOMMEND', 'bound': {'min_x': 127.04996923757218, 'min_y': 34.83903703166801, 'max_x': 128.45664949666423, 'max_y': 37.50762035337538}, 'fare': {'taxi': 360300, 'toll': 17900}, 'distance': 387814, 'duration': 16382}, 'sections': [{'distance': 387814, 'duration': 16382, 'bound': {'min_x': 128.41824916114862, 'min_y': 34.84202279070474, 'max_x': 128.44152743744485, 'max_y': 37.50447640723021}, 'roads': [{'name': '강남대로', 'distance': 1072, 'duration': 693, 'traffic_speed': 4.0, 'traffic_state': 1, 'vertexes': [127.02893499568054, 37.495438039450185, 127.02888871496768, 37.495518754020495, 127.0285162480976, 37.496335652150904, 127.02802859159864, 37.497313791202295, 127.02792378343575, 37.4975472